[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# El proceso de arranque del sistema.

El proceso de arranque (*boot process*) de un sistema *GNU/Linux* comprende una serie de etapas que van desde el encendido del equipo hasta la presentación del entorno de inicio de sesión al usuario. Conocer este proceso es fundamental para la administración y solución de problemas en sistemas *GNU/Linux*.

Las etapas principales son:

1. *Firmware* (BIOS/UEFI)
2. Gestor de arranque (*bootloader*)
3. Carga del kernel
4. *initramfs*
5. Sistema de inicio (*init* / *systemd*)
6. *Targets* y sesión de usuario

## *Firmware*: BIOS y UEFI.

Al encender el equipo, el primer software en ejecutarse es el *firmware* de la tarjeta madre.

### *BIOS* (*Basic Input/Output System*).

La *BIOS* es el *firmware* tradicional presente en equipos fabricados antes de 2010 aproximadamente. Su función principal es:

* Realizar el *POST* (*Power-On Self-Test*): verificación del hardware.
* Identificar los dispositivos de almacenamiento.
* Cargar el *MBR* (*Master Boot Record*) del disco de arranque.

### *UEFI* (*Unified Extensible Firmware Interface*).

*UEFI* es el sucesor moderno de la *BIOS*. Entre sus ventajas destacan:

* Soporte para discos mayores a 2 TB (usando *GPT* (*GUID Partition Table*)).
* Arranque seguro (*Secure Boot*).
* Interfaz gráfica y soporte de red durante el arranque.
* Tiempos de arranque más rápidos.

Los sistemas *UEFI* buscan el gestor de arranque en la *ESP* (*EFI System Partition*).

## El gestor de arranque *GRUB 2*.

*GRUB 2* (*Grand Unified Bootloader version 2*) es el gestor de arranque más utilizado en las distribuciones *GNU/Linux* modernas. Sus funciones son:

* Presentar un menú de arranque al usuario.
* Cargar el kernel de *Linux* en memoria.
* Pasar parámetros al kernel.
* Permitir el arranque dual (*dual boot*) con otros sistemas operativos.

### Archivos de configuración de *GRUB 2*.

La configuración principal se encuentra en `/boot/grub/grub.cfg`. Este archivo **no debe editarse directamente**; se genera automáticamente a partir de los archivos en `/etc/grub.d/` y `/etc/default/grub`.

**Nota sobre el entorno:** las celdas de código de este cuaderno (desde ```ls /boot/grub/``` hasta ```who -b```) consultan el arranque real del equipo: el gestor de arranque, el *kernel* en ```/boot```, ```systemctl get-default``` y la bitácora de ```journalctl -b```. Requieren una máquina *Ubuntu* 24.04 con *systemd* como proceso de identificador 1 (*PID 1*). En *GitHub Codespaces* el entorno es un contenedor: no arrancó con *GRUB* ni con *systemd*, por lo que varias de estas celdas fallan o muestran datos del contenedor. Por ejemplo, ```systemctl``` responde ```System has not been booted with systemd as init system (PID 1). Can't operate.```

La siguiente celda lista los elementos que coinciden con `/boot/grub/`.


In [ ]:
ls /boot/grub/

La siguiente celda muestra el contenido de `/etc/default/grub`.


In [ ]:
cat /etc/default/grub

### El comando ```update-grub```.

El comando `update-grub` regenera el archivo `/boot/grub/grub.cfg` a partir de los archivos de configuración. Debe ejecutarse con privilegios de superusuario tras modificar `/etc/default/grub`.

```bash
sudo update-grub
```

## El kernel y el directorio ```/boot```.

El directorio `/boot` contiene los archivos necesarios para el arranque del sistema.

La siguiente celda lista los elementos que coinciden con `/boot/`.


In [ ]:
ls /boot/

Los archivos más importantes son:

* ```vmlinuz-*```: La imagen comprimida del kernel de *Linux*.
* ```initrd.img-*```: La imagen del sistema de archivos inicial (*initramfs*).
* ```System.map-*```: Tabla de símbolos del kernel (útil para depuración).
* ```config-*```: Archivo de configuración con la que fue compilado el kernel.

La siguiente celda lista los elementos que coinciden con `/boot/vmlinuz*`.


In [ ]:
ls /boot/vmlinuz*

La siguiente celda lista los elementos que coinciden con `/boot/initrd.img*`.


In [ ]:
ls /boot/initrd.img*

## *initramfs* (sistema de archivos RAM inicial).

El *initramfs* (*initial RAM filesystem*) es un sistema de archivos temporal que el kernel monta en memoria durante el arranque. Su propósito es preparar el entorno necesario para montar el sistema de archivos raíz definitivo.

### Funciones del *initramfs*:

* Cargar los módulos del kernel necesarios para acceder al sistema de archivos raíz.
* Gestionar el descifrado de volúmenes cifrados (*LUKS*).
* Gestionar arreglos *RAID* (*Redundant Array of Independent Disks*) o volúmenes *LVM* (*Logical Volume Manager*).
* Montar el sistema de archivos raíz y transferir el control a *systemd*.

El *initramfs* es un archivo `cpio` comprimido. Para inspeccionar su contenido:

La siguiente celda ejecuta `file /boot/initrd.img-$(uname -r)`.


In [ ]:
file /boot/initrd.img-$(uname -r)

## El proceso ```init``` y *systemd*.

Una vez que el kernel ha montado el sistema de archivos raíz, ejecuta el proceso con *PID 1*, que es el proceso padre de todos los demás procesos del sistema.

En los sistemas *GNU/Linux* modernos, el proceso *PID 1* es *systemd*.

La siguiente celda ejecuta `ps -p 1`.


In [ ]:
ps -p 1

### Los *targets* de *systemd*.

Los *targets* de *systemd* son equivalentes a los *runlevels* del sistema *SysV init*. Definen el estado operativo del sistema.

| *Target* | Equivalente *SysV* | Descripción |
|---|---|---|
| `poweroff.target` | 0 | Apagado del sistema |
| `rescue.target` | 1 | Modo de rescate (usuario único) |
| `multi-user.target` | 3 | Modo multiusuario sin entorno gráfico |
| `graphical.target` | 5 | Modo multiusuario con entorno gráfico |
| `reboot.target` | 6 | Reinicio del sistema |

La siguiente celda muestra con `systemctl get-default` el *target* con el que arranca el sistema, por ejemplo `graphical.target` o `multi-user.target`. Requiere *systemd* como *PID 1* (véase la nota sobre el entorno).

In [ ]:
systemctl get-default

## Mensajes del kernel con ```dmesg```.

El comando `dmesg` muestra el *buffer* de mensajes del kernel (*kernel ring buffer*). Es muy útil para diagnosticar problemas de hardware y ver los mensajes generados durante el arranque.

La siguiente celda ejecuta `dmesg | head -20`.


In [ ]:
dmesg | head -20

La siguiente celda ejecuta `dmesg --level=err,warn | head -20`.


In [ ]:
dmesg --level=err,warn | head -20

### La man page de ```dmesg```.

La siguiente celda muestra la *man page* de `dmesg`.


In [ ]:
man dmesg

## Mensajes de arranque con ```journalctl```.

El comando `journalctl` permite consultar el diario del sistema gestionado por *systemd*. La opción `-b` muestra únicamente los mensajes del arranque actual.

La siguiente celda muestra con `journalctl -b` las primeras 30 líneas de la bitácora del arranque actual, desde los mensajes del *kernel*. Requiere *systemd* como *PID 1* (véase la nota sobre el entorno).

In [ ]:
journalctl -b | head -30

La siguiente celda ejecuta `journalctl -b -p err | head -20`.


In [ ]:
journalctl -b -p err | head -20

## Información del tiempo de arranque.

### El comando ```uptime```.

El comando `uptime` muestra el tiempo que lleva funcionando el sistema desde el último arranque, el número de usuarios conectados y la carga promedio del sistema.

La siguiente celda ejecuta `uptime`.


In [ ]:
uptime

### Hora del último arranque con ```who -b```.

El comando `who` con la opción `-b` muestra la fecha y hora del último arranque del sistema.

La siguiente celda ejecuta `who -b`.


In [ ]:
who -b

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>